# Ablation 6: Test-Time Augmentation (TTA) - Final Aggregation

**Purpose**: Final ablation. Aggregates the previous improvements:

1. **Ablation 4a** (adopted): hard negatives mixed training, 87.51% mAP50
2. **Ablation 5** (pending): label smoothing, expected +0-1% with better calibration

**Intervention**: Apply Test-Time Augmentation (TTA) at inference:
- Multiple augmentations (h-flip, v-flip, rotate 90°)
- Ensemble predictions via mean aggregation
- Expected to improve robustness on edge cases

**Expected Outcome**: +1-2% Dice improvement over best individual ablation

**Expected Runtime**: ~2-3 hours on P100 (3 seeds × 100 epochs, plus TTA inference on 552 test images)

---

## Clinical Context

TTA provides **ensemble-based uncertainty quantification**:
- Higher agreement across augmentations means higher confidence
- Lower agreement is flagged for human review
- Overconfidence is dangerous in clinical deployment

In [ ]:
# Lock NumPy to prevent ultralytics from upgrading to 2.x
import sys
print("Step 1: Record Kaggle's default NumPy version...")
import numpy as np
default_numpy = np.__version__
print(f"   Default NumPy: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: LOCK NumPy back to Kaggle's default...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

print("\nInstallation complete!")
import numpy as np
from ultralytics import YOLO
print(f"   NumPy: {np.__version__}")
print("   OK Ultralytics imported successfully")

In [ ]:
import json
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import time
import torch
from ultralytics import YOLO

# Environment detection
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# KAGGLE DATASET SETUP
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    print("[INFO] Code dataset added to Python path")
    from evaluation.metrics import calculate_segmentation_metrics, calculate_boundary_f1, calculate_bootstrap_ci
    from evaluation.yolo_utils import extract_mask_from_yolo_result
else:
    sys.path.insert(0, str(PROJECT_ROOT))
    print("[INFO] Using local src directory")
    from src.evaluation.metrics import calculate_segmentation_metrics, calculate_boundary_f1, calculate_bootstrap_ci
    from src.evaluation.yolo_utils import extract_mask_from_yolo_result

print("OK All modules imported successfully")

In [ ]:
# Configure paths
IN_KAGGLE = Path("/kaggle/input").exists()

if IN_KAGGLE:
    BASE_INPUT_PATH = Path("/kaggle/input")
    OUTPUT_PATH = Path("/kaggle/working")
    DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs/data_wound_seg"
else:
    BASE_INPUT_PATH = PROJECT_ROOT / "data"
    OUTPUT_PATH = PROJECT_ROOT / "kaggle_outputs/ablation_6_tta"
    DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs/data_wound_seg"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset: {DATASET_PATH}")
print(f"Output: {OUTPUT_PATH}")

# Verify dataset exists
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

print("OK Environment configured")

## Training Configuration

Uses the **aggregated configuration** from Ablations 4a and 5:

1. **Hard Negatives (Ablation 4a)**: Train on wound images and DDI hard negatives
2. **Label Smoothing (Ablation 5)**: `label_smoothing=0.1`
3. **P100-Optimized**: batch=32, lr=0.015, workers=8, epochs=100, patience=10

In [ ]:
# Create YOLO dataset configuration
config_path = OUTPUT_PATH / "wound_dataset.yaml"
config_content = f"""path: {DATASET_PATH.parent}
train: data_wound_seg/train
val: data_wound_seg/test

names:
  0: wound
"""

with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"OK YOLO config created: {config_path}")

# Setup test pairs for TTA evaluation
TEST_IMAGES_DIR = DATASET_PATH / "test_images"
TEST_MASKS_DIR = DATASET_PATH / "test_masks"

test_image_paths = sorted(TEST_IMAGES_DIR.glob("*.png"))
print(f"Found {len(test_image_paths)} test images")

test_pairs = []
for img_path in test_image_paths:
    mask_path = TEST_MASKS_DIR / img_path.name
    if mask_path.exists():
        test_pairs.append((str(img_path), str(mask_path)))

print(f"Valid test pairs: {len(test_pairs)}")

## Test-Time Augmentation (TTA) Implementation

TTA applies multiple augmentations at inference and ensembles predictions.

**Augmentations Used**:
1. Original (no transform)
2. Horizontal flip
3. Vertical flip
4. Rotate 90°

**Ensemble Strategy**: Mean aggregation (better than majority vote for continuous masks)

In [ ]:
# Define TTA augmentations (forward + inverse transforms)
AUGMENTATIONS = {
    "original": (
        lambda x: x,
        lambda x: x
    ),
    "horizontal_flip": (
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-1])
    ),
    "vertical_flip": (
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(x, dims=[-2])
    ),
    "rotate_90": (
        lambda x: torch.rot90(x, k=1, dims=[-2, -1]),
        lambda x: torch.rot90(x, k=-1, dims=[-2, -1])
    ),
}

print(f"OK Defined {len(AUGMENTATIONS)} augmentations for TTA")


def predict_with_tta(model, image_path, augmentation_set=None, aggregation="mean"):
    """Predict with Test-Time Augmentation."""
    if augmentation_set is None:
        augmentation_set = AUGMENTATIONS

    # Load image
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    predictions = []

    for aug_name in augmentation_set:
        forward_fn, inverse_fn = augmentation_set[aug_name]

        # Convert to tensor for augmentation
        img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float()

        # Apply augmentation
        aug_img_tensor = forward_fn(img_tensor)

        # Convert back to numpy for YOLO
        aug_img = aug_img_tensor.permute(1, 2, 0).numpy().astype(np.uint8)

        # Predict
        result = model.predict(aug_img, verbose=False)[0]
        pred_mask = extract_mask_from_yolo_result(result)

        # Resize mask if needed (BEFORE inverse transform)
        if pred_mask.shape != img_rgb.shape[:2]:
            pred_mask = cv2.resize(
                pred_mask,
                (img_rgb.shape[1], img_rgb.shape[0]),
                interpolation=cv2.INTER_NEAREST
            )

        # Convert mask to tensor for inverse transform
        pred_tensor = torch.from_numpy(pred_mask).float()
        pred_reversed = inverse_fn(pred_tensor)
        predictions.append(pred_reversed.numpy())

    # Aggregate predictions
    if aggregation == "mean":
        ensemble_mask = (np.mean(predictions, axis=0) > 0.5).astype(np.uint8)
    elif aggregation == "majority_vote":
        ensemble_mask = (np.sum(predictions, axis=0) > (len(predictions) / 2)).astype(np.uint8)
    else:
        raise ValueError(f"Unknown aggregation: {aggregation}")

    return ensemble_mask


print("OK TTA prediction function defined")

## Multi-Seed Training

Training with the aggregated configuration (Hard Negatives and Label Smoothing).

**Seeds**: [42, 123, 456]

In [ ]:
RANDOM_SEEDS = [42, 123, 456]
all_baseline_results = {}
all_tta_results = {}

for seed in RANDOM_SEEDS:
    print(f"\n{'='*80}")
    print(f"TRAINING SEED {seed}")
    print('='*80)

    # Create fresh model instance
    model = YOLO("yolov8n-seg.pt")

    start = time.time()
    results = model.train(
        data=str(config_path),
        epochs=100,
        batch=32,
        lr0=0.015,
        workers=8,
        amp=False,
        imgsz=512,
        patience=10,
        seed=seed,

        # Aggregated configuration
        label_smoothing=0.1,  # From Ablation 5

        # Baseline augmentation (disabled for clean comparison)
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
        degrees=0.0, translate=0.0, scale=0.0,
        flipud=0.0, fliplr=0.0,
        mosaic=0.0, mixup=0.0,
        auto_augment=None,

        project=str(OUTPUT_PATH / "yolo_training"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False
    )

    duration = (time.time() - start) / 60
    print(f"OK Training complete in {duration:.1f} min")

    # Load trained model
    model_path = OUTPUT_PATH / f"yolo_training/seed_{seed}/weights/best.pt"
    model = YOLO(str(model_path))

    # Evaluate with baseline (no TTA) and TTA
    print(f"\nEvaluating seed {seed}...")

    baseline_results = []
    tta_results = []
    baseline_times = []
    tta_times = []

    for img_path_str, mask_path_str in test_pairs[:50]:  # Evaluate on first 50 for speed
        image_id = Path(img_path_str).stem
        gt_mask = cv2.imread(mask_path_str, cv2.IMREAD_GRAYSCALE)
        _, gt_mask = cv2.threshold(gt_mask, 127, 1, cv2.THRESH_BINARY)

        # Baseline prediction
        start = time.time()
        img_rgb = cv2.imread(img_path_str)
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
        baseline_result = model.predict(img_rgb, verbose=False)[0]
        baseline_mask = extract_mask_from_yolo_result(baseline_result)

        if baseline_mask.shape != gt_mask.shape:
            baseline_mask = cv2.resize(baseline_mask, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)

        baseline_times.append(time.time() - start)

        # TTA prediction
        start = time.time()
        tta_mask = predict_with_tta(model, img_path_str, aggregation="mean")
        tta_times.append(time.time() - start)

        # Calculate metrics
        baseline_metrics = calculate_segmentation_metrics(baseline_mask, gt_mask, image_id)
        tta_metrics = calculate_segmentation_metrics(tta_mask, gt_mask, image_id)

        baseline_results.append(baseline_metrics)
        tta_results.append(tta_metrics)

    all_baseline_results[f"seed_{seed}"] = baseline_results
    all_tta_results[f"seed_{seed}"] = tta_results

    print(f"  OK Seed {seed}: {len(baseline_results)} images evaluated")
    print(f"  Baseline time: {np.mean(baseline_times):.3f}s per image")
    print(f"  TTA time: {np.mean(tta_times):.3f}s per image")

print("\nOK ALL TRAINING COMPLETE")

## Results: Baseline vs TTA

Model with and without Test-Time Augmentation.

In [ ]:
# Aggregate metrics across all seeds
all_baseline_pooled = []
all_tta_pooled = []

for seed in RANDOM_SEEDS:
    seed_key = f"seed_{seed}"
    all_baseline_pooled.extend(all_baseline_results[seed_key])
    all_tta_pooled.extend(all_tta_results[seed_key])

baseline_df = pd.DataFrame(all_baseline_pooled)
tta_df = pd.DataFrame(all_tta_pooled)

# Calculate bootstrap CIs
baseline_dice_ci = calculate_bootstrap_ci(
    baseline_df['dice'].tolist(), confidence_level=0.95, n_bootstrap=10000, random_seed=42
)
tta_dice_ci = calculate_bootstrap_ci(
    tta_df['dice'].tolist(), confidence_level=0.95, n_bootstrap=10000, random_seed=42
)

print("=" * 80)
print("ABLATION 6: TEST-TIME AUGMENTATION RESULTS")
print("=" * 80)

print(f"\nBaseline (no TTA):")
print(f"  Dice: {baseline_dice_ci['mean']:.4f} [{baseline_dice_ci['ci_lower']:.4f}, {baseline_dice_ci['ci_upper']:.4f}]")
print(f"  Precision: {baseline_df['precision'].mean():.4f}")
print(f"  Recall: {baseline_df['recall'].mean():.4f}")

print(f"\nWith TTA:")
print(f"  Dice: {tta_dice_ci['mean']:.4f} [{tta_dice_ci['ci_lower']:.4f}, {tta_dice_ci['ci_upper']:.4f}]")
print(f"  Precision: {tta_df['precision'].mean():.4f}")
print(f"  Recall: {tta_df['recall'].mean():.4f}")

dice_improvement = tta_dice_ci['mean'] - baseline_dice_ci['mean']
print(f"\nΔ Dice: {dice_improvement:+.4f} ({dice_improvement/baseline_dice_ci['mean']*100:+.2f}%)")

# Save final report
final_report = {
    "ablation": "6_tta",
    "baseline_dice": baseline_dice_ci["mean"],
    "baseline_ci": [baseline_dice_ci["ci_lower"], baseline_dice_ci["ci_upper"]],
    "tta_dice": tta_dice_ci["mean"],
    "tta_ci": [tta_dice_ci["ci_lower"], tta_dice_ci["ci_upper"]],
    "improvement": dice_improvement,
    "per_seed_baseline": all_baseline_results,
    "per_seed_tta": all_tta_results
}

report_path = OUTPUT_PATH / "ablation_6_final_report.json"
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"\nFinal report saved: {report_path}")

# Save completion status
status = {
    "ablation": "6_tta",
    "status": "complete",
    "seeds": RANDOM_SEEDS,
    "tta_improvement": dice_improvement
}

with open(OUTPUT_PATH / "completion_status.json", 'w') as f:
    json.dump(status, f, indent=2)

print("\nOK Ablation 6 (TTA) complete!")